# core

> the vault: one SQLite file holding everything you have read, and the retrieval over it

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

A `Vault` is one SQLite file: a litesearch document tree (docs → nodes → chunks, FTS5 + a usearch
HNSW index) with an entity graph over it. Everything you read goes in under a `kind`, and one query
crosses all of them.

In [ ]:
#| export
import json, re, time, uuid, warnings
from datetime import datetime
import numpy as np
from fastcore.all import AttrDict, L, Path, first, ifnone, patch, store_attr
from litesearch.core import database
from litesearch.graph import hash_embed, build_graph, resolve_entities, graph_stats
import litesearch.tree, litesearch.graph   # patches add_doc/context/clusters onto Database/Table

In [ ]:
#| export
KINDS = ('web', 'pdf', 'arxiv', 'youtube', 'file', 'code', 'data', 'note')

_WINDOW_NODE = re.compile(r'^Pages \d+(?:–\d+)?: ')

def tidy_bc(bc:str) -> str:
    """Drop `build_tree`'s window-node placeholders from a breadcrumb.

    A document with no usable headings — a note, a scraped page that lost its structure — gets
    nodes named `Pages 1–1: <first 60 chars>`, which is a useful internal label and pure noise in
    a citation. The node is real either way; only the display changes."""
    if not bc: return bc or ''
    parts = [p.strip() for p in bc.split('›')]
    keep = [p for p in parts if not _WINDOW_NODE.match(p)]
    out, seen = [], set()
    for p in (keep or parts):                       # never return empty: some node must be named
        if p and p not in seen: seen.add(p); out.append(p)
    return ' › '.join(out)

def _ts(v) -> float:
    'Best-effort epoch seconds from a float, an int, or a SQLite CURRENT_TIMESTAMP string.'
    if isinstance(v, (int, float)): return float(v)
    if isinstance(v, str):
        try: return datetime.fromisoformat(v.replace('Z', '+00:00')).timestamp()
        except ValueError: return 0.0
    return 0.0

In [ ]:
#| export
def mk_encoder(model:str=None,      # model2vec/HF id; None -> the retrieval default
               dims:int=256,        # dims for the hashing fallback only
               offline:bool=False,  # skip the download attempt entirely
) -> AttrDict:
    """The best text encoder available, degrading to a hashing embedder rather than failing.

    Returns `AttrDict(doc, query, dtype, dims, method, note)`. `method`/`note` follow litesearch's
    convention that a degradable call reports which backend answered and why. That matters more
    here than usual: the gap between real semantics and char-n-gram hashing is the gap between a
    vault that answers questions and one that only does keyword search, so it must never be silent."""
    if not offline:
        nm = model or 'minishlab/potion-retrieval-32M'
        try:
            from litesearch.utils import static_embedder
            m = static_embedder(nm)
            enc = lambda ts, **kw: m.encode(list(ts))
            v = enc(['probe'])
            return AttrDict(doc=enc, query=enc, dtype=v.dtype.type, dims=int(v.shape[-1]),
                            method='model2vec', note=f'{nm} ({v.shape[-1]}d, {v.dtype})')
        except Exception as e:
            warnings.warn(f'could not load {nm} ({type(e).__name__}: {str(e)[:120]}); '
                          f'falling back to hash_embed — retrieval will be lexical, not semantic')
    enc = lambda ts, **kw: hash_embed(ts, ndim=dims, dtype=np.float16)
    return AttrDict(doc=enc, query=enc, dtype=np.float16, dims=dims, method='hash',
                    note=f'char-n-gram hashing ({dims}d) — no model available, so hits are '
                         f'lexical; pass encoder= or restore network access for real semantics')

In [ ]:
#| export
class Vault:
    """Everything you have read, in one SQLite file, searchable as one corpus.

    A vault is a `litesearch` document tree (docs → nodes → chunks, with FTS5 and a usearch HNSW
    index) plus an entity graph over it. Web pages, PDFs, papers, transcripts, local files, code
    and your own notes all land in the same store under different `kind`s, which is the whole
    point: one query crosses all of them, and `context()` hands back sections rather than fragments.

    Acquisition (`vault.web/url/arxiv/youtube/...`) lives in `vishalakshi.acquire`; answering
    (`vault.ask`) in `vishalakshi.ask`. Both are optional imports — the vault itself needs neither
    a network nor an LLM."""

    def __init__(self,
                 path:str=None,       # vault file; None -> ~/.vishalakshi/vault.db
                 encoder=None,        # AttrDict from mk_encoder(), or a model id, or None
                 store:str='store',   # chunk store name
                 offline:bool=False,  # never attempt a model download
                 dims:int=256):       # dims for the hashing fallback
        self.path = str(ifnone(path, Path.home()/'.vishalakshi'/'vault.db'))
        self.store = store
        self.enc = (encoder if isinstance(encoder, AttrDict)
                    else mk_encoder(encoder, dims=dims, offline=offline))
        self.db = database(self.path)
        self.g = self.db.get_tree(store, dtype=self.enc.dtype, ndim=self.enc.dims)

    def __repr__(self):
        s = self.stats()
        return (f"Vault({self.path!r}: {s['docs']} docs, {s['chunks']} chunks, "
                f"{s['entities']} entities, encoder={self.enc.method})")

    # ---- embedding -------------------------------------------------------------------------
    def embed(self, texts, **kw):
        'Document-side vectors for `texts`.'
        return self.enc.doc(list(texts), **kw)

    def qv(self, q:str) -> bytes:
        'Query-side embedding of `q`, as the bytes every litesearch search call wants.'
        return np.asarray(self.enc.query([q])[0], dtype=self.enc.dtype).tobytes()

    @property
    def dtype(self): return self.enc.dtype

In [ ]:
#| export
@patch
def add(self:Vault,
        pages,                # markdown/text, or [(page_no, text)]
        title:str,            # document title
        source:str=None,      # url or path; defaults to the title. Identity is hashed over it
        kind:str='file',      # one of KINDS — the facet you filter and report on
        meta:dict=None,       # provenance: the query that found it, when, which tier fetched it
        force:bool=False,     # re-ingest a source already present
        **kw                  # forwarded to litesearch add_doc (chunker, summarize, with_heading)
) -> dict:
    """Ingest one document into the vault: tree, chunks, embeddings, ANN index.

    Identity is content-addressed over `source|title`, so re-adding the same page is a no-op rather
    than a duplicate — which is what makes it safe to re-run a search whose results overlap what
    you already have."""
    meta = dict(meta or {}, added_at=time.time())
    return self.db.add_doc(pages, title, source=source, kind=kind, store=self.store,
                           emb_fn=self.enc.doc, meta=meta, force=force, **kw)

@patch
def assets(self:Vault, name:str=None) -> Path:
    'Where extracted assets (PDF images) go: beside the vault file, never the working directory.'
    d = Path(self.path).parent/'assets'
    d.mkdir(parents=True, exist_ok=True)
    return d/name if name else d

@patch
def add_file(self:Vault, path, title:str=None, kind:str=None, **kw) -> dict:
    """Ingest one local file (PDFs page by page; notebooks through ipynb_parse).

    PDFs are parsed here rather than through `litesearch.add_file` for one reason: pdf-oxide writes
    extracted images relative to its `out_path`, which defaults to `./pdfs/`, so ingesting a paper
    would silently litter whatever directory you happened to be in. They go next to the vault."""
    p = Path(path)
    ttl = title or p.stem.replace('_', ' ').replace('-', ' ').strip()
    if p.suffix.lower() == '.pdf':
        from litesearch.data import pdf_parse
        pages = list(enumerate(pdf_parse(str(p), out_path=self.assets(p.stem))))
        return self.add(pages, ttl, source=str(p), kind=kind or 'pdf', **kw)
    k = kind or 'file'
    r = self.db.add_file(p, title=title, store=self.store, emb_fn=self.enc.doc, **kw)
    if r.get('doc_id') and not r.get('skipped'):
        self.g.docs.update(dict(id=r['doc_id'], kind=k, meta=json.dumps(dict(added_at=time.time()))))
    return r

@patch
def add_dir(self:Vault, dir, types:str=None, **kw) -> list:
    'Ingest every document under a directory. Already-ingested sources are skipped, not duplicated.'
    from litesearch.tree import DOC_EXTS
    exts = {f".{t.strip().lstrip('.')}".lower() for t in (types or DOC_EXTS).split(',')}
    return [self.add_file(p, **kw) for p in sorted(Path(dir).rglob('*'))
            if p.is_file() and p.suffix.lower() in exts]

@patch
def _set_kind(self:Vault, did:str, kind:str):
    'Stamp a doc row with its kind after the fact (add_file infers kind from the extension).'
    self.g.docs.update(dict(id=did, kind=kind))

@patch
def note(self:Vault,
         text:str,            # what you want to remember
         title:str=None,      # defaults to the first line
         tags:list=None,      # free-form tags, kept in the doc's meta
) -> dict:
    """Write a note into the vault so it is searched alongside the corpus.

    Notes are ordinary documents with `kind='note'`, which is deliberate: the graph, the clusters
    and `context()` all see them for free, so what you concluded about a corpus comes back next to
    the evidence you concluded it from."""
    ttl = title or (text.strip().splitlines() or ['note'])[0].lstrip('# ')[:80]
    return self.add(text.strip(), ttl, source=f'note:{uuid.uuid4().hex[:12]}',
                    kind='note', meta=dict(tags=list(tags or [])))

In [ ]:
#| export
@patch
def _kind_docs(self:Vault, kind):
    'Doc ids matching a kind filter (str or iterable), or None for no filter.'
    if not kind: return None
    ks = [kind] if isinstance(kind, str) else list(kind)
    return {r['id'] for r in self.g.docs() if r['kind'] in ks}

@patch
def find(self:Vault,
         q:str,              # query
         limit:int=10,       # hits to return
         kind=None,          # restrict to one or more KINDS
         **kw                # forwarded to litesearch doc_search
) -> list:
    'Chunk-level hybrid search (FTS5 + vectors, RRF-fused), each hit carrying its breadcrumb.'
    keep = self._kind_docs(kind)
    hits = self.db.doc_search(q, self.qv(q), limit=limit*4 if keep else limit,
                              store=self.store, dtype=self.dtype, **kw)
    if keep is not None: hits = [h for h in hits if h.get('doc_id') in keep]
    for h in hits: h['breadcrumb'] = tidy_bc(h.get('breadcrumb'))
    return hits[:limit]

@patch
def sections(self:Vault, q:str, limit:int=5, kind=None, per:int=3, **kw) -> list:
    'Ranked *sections* rather than chunks — the unit worth reading, each with a `read` handle.'
    keep = self._kind_docs(kind)
    secs = self.db.sections(q, self.qv(q), limit=limit*4 if keep else limit, per=per,
                            store=self.store, dtype=self.dtype, **kw)
    if keep is not None:
        nodes = {r['id']: r['doc_id'] for r in self.g.nodes()}
        secs = [s for s in secs if nodes.get(s['node_id']) in keep]
    for s in secs: s['breadcrumb'] = tidy_bc(s.get('breadcrumb'))
    return secs[:limit]

@patch
def context(self:Vault,
            q:str,              # the question
            sections:int=6,     # operative sections returned
            related:int=8,      # related sections reached by graph + vector
            kind=None,          # restrict to one or more KINDS
            max_read:int=6000,  # chars of assembled text per section
            **kw                # forwarded to litesearch context
) -> AttrDict:
    """The retrieval an LLM should be handed: whole sections plus what they connect to.

    Wraps `litesearch.Database.context`. Operative sections carry `text, breadcrumb, pages,
    filename` and their tree neighbourhood; `related` holds sections reached by the entity graph
    (`via='graph'`) and by embedding similarity (`via='vector'`). Pass `kind=` to scope it to, say,
    only your notes or only the papers.

    `kind` filters after retrieval rather than inside it, because litesearch's `context` does not
    thread a `where` clause down to both legs. The over-fetch below covers the common case; a
    filter matching very little of a large vault can still come back short."""
    keep = self._kind_docs(kind)
    ctx = self.db.context(q, self.qv(q), store=self.store,
                          sections=sections*3 if keep else sections, related=related,
                          max_read=max_read, **kw)
    if keep is not None:
        ctx.results = L([r for r in ctx.results if r.doc_id in keep])[:sections]
        ctx.related = L([r for r in ctx.related if r.doc_id in keep])[:related]
    for r in (*ctx.results, *ctx.related): r.breadcrumb = tidy_bc(r.breadcrumb)
    ctx.encoder = self.enc.note
    return ctx

@patch
def related(self:Vault, node_id:str, limit:int=8) -> list:
    """Sections nearest an existing one — "what else in the vault reads like this".

    Reuses the vectors usearch already holds, so nothing is re-embedded."""
    rows = self.g.store(select='rowid as rowid, node_id', where=f'node_id={node_id!r}')
    if not rows: return []
    seen, out = {node_id}, []
    for r in rows:
        for n in self.g.store.ann_neighbors(r['rowid'], limit=limit*3, dtype=self.dtype,
                                            columns=['content','node_id','doc_id']):
            nid = n.get('node_id')
            if not nid or nid in seen: continue
            seen.add(nid)
            out.append(dict(node_id=nid, doc_id=n.get('doc_id'), dist=n.get('_dist'),
                            breadcrumb=tidy_bc(self.db.breadcrumb(nid, self.store)),
                            snippet=(n.get('content') or '')[:300]))
            if len(out) >= limit: return out
    return out

@patch
def read(self:Vault, node_id:str, max_chars:int=6000) -> dict:
    'Assemble a whole section back out of its chunks.'
    return self.db.read(node_id, store=self.store, max_chars=max_chars)

@patch
def toc(self:Vault, **kw) -> list:
    'The table of contents across every document in the vault.'
    return self.db.toc(store=self.store, **kw)

In [ ]:
#| export
@patch
def connect(self:Vault, resolve:bool=True, **kw) -> dict:
    """(Re)build the entity graph over everything in the vault.

    This is what makes `context()`'s `via='graph'` leg work: sections that share no vocabulary with
    the query but are reachable along an entity path. Run it after a batch of ingests rather than
    per document — it reads the whole store."""
    chunks = list(self.g.store())
    if not chunks: return dict(entities=0, mentions=0, edges=0, windows=0)
    res = build_graph(self.db, chunks, store=self.store, emb_fn=self.enc.doc, **kw)
    if resolve: res = dict(res, resolved=resolve_entities(self.db, store=self.store, dtype=self.dtype))
    return res

@patch
def map(self:Vault, min_count:int=2, columns:list=None) -> AttrDict:
    'Cluster the corpus into labelled topics — the shape of what you have collected.'
    return self.g.store.clusters(min_count=min_count, dtype=self.dtype,
                                 columns=columns or ['content','doc_id'])

@patch
def sources(self:Vault, kind=None) -> list:
    'Every document in the vault with its provenance, newest first.'
    ks = None if not kind else ([kind] if isinstance(kind, str) else list(kind))
    out = []
    for d in self.g.docs():
        if ks and d['kind'] not in ks: continue
        try: meta = json.loads(d['meta'] or '{}')
        except Exception: meta = {}
        out.append(dict(doc_id=d['id'], title=d['title'], kind=d['kind'], source=d['source'],
                        pages=d['pages'], added_at=_ts(meta.get('added_at') or d['added_at']),
                        meta=meta))
    return sorted(out, key=lambda r: -r['added_at'])

@patch
def forget(self:Vault, doc_id:str):
    'Remove a document, its sections and its chunks, and rebuild the ANN index.'
    self.db.delete_doc(doc_id, store=self.store)

@patch
def stats(self:Vault) -> dict:
    'Row counts across the vault, by kind.'
    kinds, docs = {}, list(self.g.docs())
    for d in docs: kinds[d['kind']] = kinds.get(d['kind'], 0) + 1
    ents = 0
    try: ents = graph_stats(self.db, store=self.store)['entities']
    except Exception: pass
    return dict(docs=len(docs), nodes=len(list(self.g.nodes())), chunks=len(list(self.g.store())),
                entities=ents, by_kind=kinds, encoder=self.enc.method, path=self.path)